# Server execution: quality first, then three-task nested tuning

Follow `README.md`. The protocol remains five outer and four inner patient-grouped
folds, seed 1729. It is not leave-one-patient-out. All three tasks produce 15 outer
jobs. The server launcher runs the regression checks before tuning. No notebook
cell starts training.

```bash
source ./setup_environment.sh
CPUSET=224-255 ./run_all.sh --data-quality-only --output results/revision-quality-v3
# Inspect Data-Quality.ipynb before continuing.
CPUSET=224-255 ./run_all.sh --quality-results results/revision-quality-v3 --output results/revision-three-tasks-v3
python -m revision.report_cli check --results-dir results/revision-three-tasks-v3 --verify-inputs
```

Use `CPUSET=0-23` for a 24-CPU allocation. Use `--resume` only for an interrupted or
failed run with its original output directory and unchanged configuration/inputs.
Old four-task result directories are preserved but incompatible with this protocol.


In [1]:
import json, os
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown, Image, FileLink

REPO = Path.cwd()
if not (REPO / "run_revision.py").is_file():
    REPO = REPO.parent
assert (
    REPO / "run_revision.py"
).is_file(), "Start Jupyter in the repository or manuscript directory"
import sys

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from revision.plot_style import apply_style, color_palette_full

apply_style()

from revision.data import TASK_PROTOCOL, TASK_TITLES, TASK3_CAPTION


def csv(path, limit=None):
    path = Path(path)
    if not path.is_file():
        display(Markdown(f"Pending/unavailable: `{path.name}`"))
        return pd.DataFrame()
    try:
        table = pd.read_csv(path)
    except pd.errors.EmptyDataError:
        table = pd.DataFrame()
    display(table if limit is None else table.head(limit))
    return table


def images(paths):
    for path in paths:
        display(Markdown(f"**{path.name}**"))
        rendered = (
            REPO / "results/revision-palette" / path.relative_to(REPO / "results")
        )
        display(Image(filename=str(rendered if rendered.is_file() else path)))
        if "cancer_vs_lr_hr" in str(path):
            display(Markdown(TASK3_CAPTION.replace("\n", " ")))


config = json.loads((REPO / "configs/revision.json").read_text())
assert config["task_protocol"] == TASK_PROTOCOL
display(config)

{'schema_version': 3,
 'seed': 1729,
 'outer_folds': 5,
 'inner_folds': 4,
 'outer_repeats': 1,
 'tasks': ['healthy_vs_cancer', 'highrisk_vs_lowrisk', 'cancer_vs_lr_hr'],
 'models': ['svm', 'rf', 'lr'],
 'objectives': ['auc', 'acc', 'f1', 'sensitivity'],
 'top_k': 1,
 'default_cutoff': 60,
 'cutoffs': [80, 100.000001],
 'normalizations': ['global', 'freq'],
 'imputations': ['mean'],
 'class_weights': [None, 'balanced'],
 'patient_weighted_training': True,
 'masks': ['All',
  'Long a+',
  'Long a+ ext.',
  'Med. a+ ext.',
  'Skip1 close',
  'Adj. close',
  'Med. adj.',
  'Adj. far',
  'Skip1 medium',
  'Skip1 far',
  'Opp. close',
  'Opp. medium',
  'Opp. far'],
 'z_quantiles': [0.1, 0.25, 0.5, 0.75, 0.9],
 'selectors': ['pca', 'mutual_info'],
 'comparison_frequencies': 3,
 'representations': ['magnitude', 'magnitude_phase'],
 'max_pca_rows': 100000,
 'calibration': True,
 'sensitivity_target': 0.95,
 'specificity_targets': [0.8, 0.9, 0.95],
 'prevalences': [0.1, 0.2],
 'bootstrap_sampl

## Selection and reporting

Each outer fold independently screens preprocessing, PCA/MI ranking, frequency
counts, geometric/magnitude IIVV masks, representations and weighting within four
inner patient folds. SVM, RF and LR are tuned for AUC, accuracy, F1 and sensitivity.
Calibration and binary operating thresholds use only outer-training OOF predictions.
Outer-test patients never rank candidates. Failed fits are recorded, and incomplete
model streams have no complete-partition pooled estimate.

Use `Data-Quality.ipynb`, `Revision-Snapshot.ipynb`, `Figures-and-Tables.ipynb`,
and `manuscript/Reviewer-Evidence.ipynb` for the corresponding saved evidence.
